# KG1 V203 - Public Intel and Adapter Triage

Objetivo: subir ranking sem destruir o baseline V194.

Este notebook faz inventario publico de Hugging Face, Kaggle e OpenRouter, prepara uma fila de adapters/datasets para triagem, e fornece helpers de avaliacao estrita contra V194.

Seguranca operacional:
- Nao faz Kaggle submit.
- Nao treina por padrao.
- Nao baixa adapters HF por padrao.
- Nao chama OpenRouter por padrao.
- Qualquer candidato precisa vencer V194 em `all720` e `official360` antes de virar submit candidate.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

IS_COLAB = Path('/content').exists()
WORK_DIR = Path('/content/kg1_v203') if IS_COLAB else Path.cwd() / '.kg1_v203'
SCRIPT_DIR = WORK_DIR / 'scripts'
OUT_DIR = Path(os.environ.get('KG1_V203_OUT', '/content/drive/MyDrive/KG1_NVIDIA_V203/public_intel_triage' if IS_COLAB else str(WORK_DIR / 'out')))
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

V194_ADAPTER = Path('/content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter')
V194_ADAPTER_SHA256 = '01259fef943bc16c31d8f7907be076cc987381a6a1bbe732b1b33c2d9f2ea95f'
V194_ALL720 = 0.16335251388243502
V194_OFFICIAL360 = 0.12171823230261604

ARCHIVE_ZIP = Path('/content/kg1_v202d/data/tonghuikang-0-87-nemotron-dataset.zip')
ARCHIVE_SHA256 = '461776d6bc44d482988d23c4e584128b66a93d2500fe7c428f4e895ab42e9eb8'
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
MODEL_REVISION = 'cbd3fa9f933d55ef16a84236559f4ee2a0526848'

print('WORK_DIR:', WORK_DIR)
print('OUT_DIR:', OUT_DIR)
print('V194_ADAPTER:', V194_ADAPTER)
print('NO KAGGLE SUBMIT IN THIS NOTEBOOK')


In [ ]:
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped/failed:', repr(exc))

if V194_ADAPTER.exists():
    adapter_model = V194_ADAPTER / 'adapter_model.safetensors'
    print('V194 adapter exists:', V194_ADAPTER)
    if adapter_model.exists():
        got = sha256_file(adapter_model)
        print('V194 adapter_model sha256:', got)
        assert got == V194_ADAPTER_SHA256, 'Unexpected V194 adapter sha256. Stop.'
else:
    print('V194 adapter not found yet. Mount Drive or adjust V194_ADAPTER before eval.')


## Baixar scripts oficiais do repo

A celula abaixo tenta primeiro a branch `claude/competent-shamir`, depois fallbacks. Ela baixa somente scripts de avaliacao, gate e preflight. Nao executa treino nem submissao.

In [ ]:
RAW_SOURCES = [
    ('FELIPEACASTRO/KG1', 'claude/competent-shamir', 'scripts'),
    ('FELIPEACASTRO/KG1-NVIDIA', 'claude/competent-shamir', 'scripts'),
    ('FELIPEACASTRO/KG1-NVIDIA', 'master', 'competent-shamir/scripts'),
    ('FELIPEACASTRO/KG1', 'master', 'scripts'),
]

SCRIPT_NAMES = [
    'hf_job_train_v90.py',
    'kg1_v202_pretokenized_adapter_eval.py',
    'kg1_submission_gate.py',
    'nemotron_submission_preflight.py',
    'kg1_convert_local_training_adapter_to_kaggle_zip.py',
]

def download_script(name, required=True):
    dest = SCRIPT_DIR / name
    last_err = None
    for repo, ref, prefix in RAW_SOURCES:
        url = f'https://raw.githubusercontent.com/{repo}/{ref}/{prefix}/{name}'
        print('Downloading:', url)
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                data = r.read()
            dest.write_bytes(data)
            print('OK:', dest, 'bytes=', len(data))
            return dest
        except Exception as exc:
            last_err = exc
    if required:
        raise RuntimeError(f'Failed to download {name}: {last_err}')
    print('WARN: failed optional script', name, repr(last_err))
    return None

downloaded = {}
for name in SCRIPT_NAMES:
    downloaded[name] = str(download_script(name, required=name in ['hf_job_train_v90.py', 'kg1_v202_pretokenized_adapter_eval.py']))

EVAL_SCRIPT = SCRIPT_DIR / 'kg1_v202_pretokenized_adapter_eval.py'
TRAIN_SCRIPT = SCRIPT_DIR / 'hf_job_train_v90.py'
print(json.dumps(downloaded, indent=2))


## V203 inventario publico

Consulta APIs publicas sem token quando possivel. Datasets gated do HF podem aparecer no inventario, mas o download real precisa de aceite de termos e token HF.

In [ ]:
def http_json(url, headers=None, timeout=90):
    req = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode('utf-8'))

def try_json(url, headers=None):
    try:
        return {'ok': True, 'url': url, 'data': http_json(url, headers=headers)}
    except Exception as exc:
        return {'ok': False, 'url': url, 'error': repr(exc)}

hf_headers = {}
hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
if hf_token:
    hf_headers['Authorization'] = f'Bearer {hf_token}'

queries = {
    'hf_datasets_reasoning_challenge': 'https://huggingface.co/api/datasets?search=nemotron%20reasoning%20challenge&limit=50',
    'hf_datasets_nemotron_kaggle': 'https://huggingface.co/api/datasets?search=NVIDIA%20Nemotron%20Kaggle&limit=50',
    'hf_models_lora_nemotron': 'https://huggingface.co/api/models?search=nemotron%2030b%20lora%20reasoning&limit=50',
    'hf_models_base_adapter': 'https://huggingface.co/api/models?other=base_model:adapter:nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16&limit=50',
    'openrouter_models': 'https://openrouter.ai/api/v1/models',
}

inventory = {
    'generated_at': utc_now(),
    'env': {
        'hf_token_present': bool(hf_token),
        'openrouter_key_present': bool(os.environ.get('OPENROUTER_API_KEY')),
        'kaggle_credentials_present': bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')),
    },
    'queries': {},
    'known_public_refs': {
        'hf_datasets': [
            'andy279/nemotron-reasoning-challenge',
            'andy279/nemotron-reasoning-challenge-raw-traces',
            'jasonkung98/NVIDIA-Nemotron-Model-Reasoning-Challenge',
        ],
        'hf_adapter_candidates': [
            'etencore/nemotron-30b-reasoning-lora',
            'Aitherium/Nemotron-3-Nano-30B-LoRA-Reasoning-v2',
            'gfinin/nemotron-reasoning-lora',
            'U2DIA/nemotron-v14-epoch1',
        ],
        'kaggle_public_refs': [
            'https://www.kaggle.com/datasets/sebmontreal/nvidia-nemotron-model-reasoning-challenge',
            'https://www.kaggle.com/code/hosseinbadrnezhad/nvidia-nemotron-high-speed-rank-32-reasoning-base/comments',
            'https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge',
        ],
    },
}

for name, url in queries.items():
    print('Query:', name)
    headers = hf_headers if name.startswith('hf_') else None
    inventory['queries'][name] = try_json(url, headers=headers)

inventory_path = OUT_DIR / 'V203_public_intel_inventory.json'
inventory_path.write_text(json.dumps(inventory, indent=2, sort_keys=True), encoding='utf-8')
print('Wrote:', inventory_path)


In [ ]:
inventory = json.loads((OUT_DIR / 'V203_public_intel_inventory.json').read_text(encoding='utf-8'))

def compact_hf(items, max_items=20):
    out = []
    if isinstance(items, dict):
        items = items.get('data') or []
    for item in (items or [])[:max_items]:
        out.append({
            'id': item.get('id') or item.get('modelId'),
            'downloads': item.get('downloads'),
            'lastModified': item.get('lastModified'),
            'tags': [t for t in (item.get('tags') or []) if any(k in str(t).lower() for k in ['adapter', 'safetensors', 'nemotron', 'reasoning', 'kaggle', 'base_model'])][:8],
        })
    return out

summary = {'generated_at': utc_now(), 'datasets': {}, 'models': {}, 'openrouter_relevant': []}
for key, payload in inventory['queries'].items():
    if not payload.get('ok'):
        continue
    data = payload.get('data')
    if key.startswith('hf_datasets'):
        summary['datasets'][key] = compact_hf(data)
    elif key.startswith('hf_models'):
        summary['models'][key] = compact_hf(data)
    elif key == 'openrouter_models':
        rows = data.get('data') or []
        wanted = ['gpt-5.5', 'claude-opus-4.7', 'claude-sonnet-4.6', 'deepseek-v4', 'qwen3.6', 'grok-4.20', 'grok-4.3', 'nemotron-3', 'owl-alpha']
        for m in rows:
            blob = ((m.get('id') or '') + ' ' + (m.get('name') or '')).lower()
            if any(w in blob for w in wanted):
                summary['openrouter_relevant'].append({
                    'id': m.get('id'),
                    'name': m.get('name'),
                    'context_length': m.get('context_length'),
                    'pricing': m.get('pricing'),
                })

summary_path = OUT_DIR / 'V203_public_intel_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
print('Wrote:', summary_path)
print(json.dumps(summary, indent=2)[:6000])


## Descoberta Kaggle opcional

Ative somente se `KAGGLE_USERNAME` e `KAGGLE_KEY` estiverem configurados. Esta celula lista datasets/notebooks; nao submete nada.

In [ ]:
RUN_KAGGLE_DISCOVERY = False

def run_cmd(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    p = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-4000:])
    return {'returncode': p.returncode, 'stdout': p.stdout}

if RUN_KAGGLE_DISCOVERY:
    if not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
        raise RuntimeError('Set KAGGLE_USERNAME and KAGGLE_KEY first.')
    try:
        import kaggle  # noqa: F401
    except Exception:
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
    results = {
        'generated_at': utc_now(),
        'datasets_nemotron': run_cmd(['kaggle', 'datasets', 'list', '-s', 'nemotron reasoning challenge', '--csv']),
        'kernels_nemotron': run_cmd(['kaggle', 'kernels', 'list', '-s', 'NVIDIA Nemotron reasoning adapter', '--csv']),
        'submissions': run_cmd(['kaggle', 'competitions', 'submissions', '-c', 'nvidia-nemotron-model-reasoning-challenge']),
    }
    out = OUT_DIR / 'V203_kaggle_discovery.json'
    out.write_text(json.dumps(results, indent=2, sort_keys=True), encoding='utf-8')
    print('Wrote:', out)
else:
    print('Skipped. Set RUN_KAGGLE_DISCOVERY=True to run.')


## V204 adapter triage opcional

Baixa adapters HF candidatos e avalia contra V194. Por padrao fica desligado para evitar downloads grandes e execucao H100/A100 sem querer.

In [ ]:
RUN_HF_ADAPTER_DOWNLOAD = False
HF_ADAPTER_IDS = [
    'etencore/nemotron-30b-reasoning-lora',
    'Aitherium/Nemotron-3-Nano-30B-LoRA-Reasoning-v2',
    'gfinin/nemotron-reasoning-lora',
    'U2DIA/nemotron-v14-epoch1',
]
HF_ADAPTER_ROOT = OUT_DIR / 'hf_adapter_candidates'
HF_ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)

downloaded_adapters = []
if RUN_HF_ADAPTER_DOWNLOAD:
    try:
        from huggingface_hub import snapshot_download
    except Exception:
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
        from huggingface_hub import snapshot_download
    for repo_id in HF_ADAPTER_IDS:
        print('Downloading HF adapter:', repo_id)
        local_dir = HF_ADAPTER_ROOT / repo_id.replace('/', '__')
        try:
            path = snapshot_download(repo_id=repo_id, local_dir=str(local_dir), token=hf_token, ignore_patterns=['*.md', '*.txt'])
            downloaded_adapters.append({'repo_id': repo_id, 'path': path})
            print('OK:', path)
        except Exception as exc:
            downloaded_adapters.append({'repo_id': repo_id, 'error': repr(exc)})
            print('FAILED:', repo_id, repr(exc))
    out = OUT_DIR / 'V204_hf_adapter_downloads.json'
    out.write_text(json.dumps(downloaded_adapters, indent=2, sort_keys=True), encoding='utf-8')
    print('Wrote:', out)
else:
    print('Skipped. Set RUN_HF_ADAPTER_DOWNLOAD=True to run.')


In [ ]:
SPLIT_SPECS_BOTH = [
    {'name': 'all720', 'val_examples': 720, 'eval_max_examples': 720, 'exclude_categories': ''},
    {'name': 'official360', 'val_examples': 360, 'eval_max_examples': 360, 'exclude_categories': 'matching,concatenation,splitting,spelling,lstrip'},
]
SPLIT_SPECS_OFFICIAL_ONLY = [SPLIT_SPECS_BOTH[1]]

def eval_adapter(label, adapter_dir, split_specs=SPLIT_SPECS_OFFICIAL_ONLY):
    adapter_dir = Path(adapter_dir)
    if not adapter_dir.exists():
        raise FileNotFoundError(adapter_dir)
    if not ARCHIVE_ZIP.exists():
        raise FileNotFoundError(f'Missing archive zip: {ARCHIVE_ZIP}')
    report = OUT_DIR / 'eval_reports' / f'{label}_eval.json'
    report.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, '-u', str(EVAL_SCRIPT),
        '--training-script', str(TRAIN_SCRIPT),
        '--adapter-dir', str(adapter_dir),
        '--archive-zip', str(ARCHIVE_ZIP),
        '--expected-archive-sha256', ARCHIVE_SHA256,
        '--output-json', str(report),
        '--label', label,
        '--split-specs-json', json.dumps(split_specs),
        '--model-name', MODEL_NAME,
        '--model-revision', MODEL_REVISION,
        '--model-device-map', 'auto',
        '--max-length', '8192',
        '--seed', '202',
    ]
    result = run_cmd(cmd)
    payload = {'label': label, 'adapter_dir': str(adapter_dir), 'returncode': result['returncode'], 'report': str(report)}
    if report.exists():
        payload['eval'] = json.loads(report.read_text(encoding='utf-8'))
    return payload

def compare_vs_v194(eval_payload):
    split_losses = {}
    for split in eval_payload.get('eval', {}).get('splits', []):
        split_losses[split['name']] = split['overall_loss']
    return {
        'label': eval_payload.get('label'),
        'official360': split_losses.get('official360'),
        'all720': split_losses.get('all720'),
        'official360_delta': None if split_losses.get('official360') is None else split_losses['official360'] - V194_OFFICIAL360,
        'all720_delta': None if split_losses.get('all720') is None else split_losses['all720'] - V194_ALL720,
        'passed_official360': split_losses.get('official360') is not None and split_losses['official360'] < V194_OFFICIAL360,
        'passed_all720': split_losses.get('all720') is not None and split_losses['all720'] < V194_ALL720,
    }

print('Helper ready. First run official360 only. Promote to both splits only if official360 beats V194.')


In [ ]:
RUN_STRICT_ADAPTER_TRIAGE = False

if RUN_STRICT_ADAPTER_TRIAGE:
    candidates = []
    downloads_path = OUT_DIR / 'V204_hf_adapter_downloads.json'
    if downloads_path.exists():
        for row in json.loads(downloads_path.read_text(encoding='utf-8')):
            if row.get('path'):
                candidates.append((row['repo_id'].replace('/', '__'), row['path']))
    if not candidates:
        raise RuntimeError('No downloaded adapters. Run HF download cell first or add candidates manually.')
    results = []
    for label, path in candidates:
        official = eval_adapter(label + '_official360', path, SPLIT_SPECS_OFFICIAL_ONLY)
        cmp_official = compare_vs_v194(official)
        results.append({'stage': 'official360', 'comparison': cmp_official, 'payload': official})
        if cmp_official['passed_official360']:
            both = eval_adapter(label + '_both', path, SPLIT_SPECS_BOTH)
            results.append({'stage': 'both', 'comparison': compare_vs_v194(both), 'payload': both})
        else:
            print('Reject after official360:', label, cmp_official)
    out = OUT_DIR / 'V204_adapter_triage_results.json'
    out.write_text(json.dumps(results, indent=2, sort_keys=True), encoding='utf-8')
    print('Wrote:', out)
else:
    print('Skipped. Set RUN_STRICT_ADAPTER_TRIAGE=True to run on H100/A100.')


## Painel OpenRouter opcional

Usar apenas para auditoria de plano e filtros. Nao usar API externa na submissao Kaggle. Por padrao chama somente modelos gratuitos e exige `OPENROUTER_API_KEY`.

In [ ]:
RUN_OPENROUTER_PANEL = False
ALLOW_PAID_OPENROUTER = False
OPENROUTER_MODELS = [
    'openrouter/owl-alpha',
    'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free',
    'nvidia/nemotron-3-super-120b-a12b:free',
    'nvidia/nemotron-3-nano-30b-a3b:free',
]
PAID_OPENROUTER_MODELS = [
    'openai/gpt-5.5',
    'anthropic/claude-opus-4.7',
    'deepseek/deepseek-v4-pro',
    'qwen/qwen3.6-flash',
]

def openrouter_chat(model_id, prompt, max_tokens=900):
    key = os.environ.get('OPENROUTER_API_KEY')
    if not key:
        raise RuntimeError('OPENROUTER_API_KEY missing')
    payload = {
        'model': model_id,
        'messages': [
            {'role': 'system', 'content': 'You are a terse ML competition risk auditor. Return compact JSON only.'},
            {'role': 'user', 'content': prompt},
        ],
        'temperature': 0.1,
        'max_tokens': max_tokens,
    }
    req = urllib.request.Request(
        'https://openrouter.ai/api/v1/chat/completions',
        data=json.dumps(payload).encode('utf-8'),
        headers={
            'Authorization': f'Bearer {key}',
            'Content-Type': 'application/json',
            'HTTP-Referer': 'https://github.com/FELIPEACASTRO/KG1',
            'X-Title': 'KG1 V203 public intel triage',
        },
    )
    with urllib.request.urlopen(req, timeout=180) as r:
        return json.loads(r.read().decode('utf-8'))

audit_prompt = f'''
We are in the NVIDIA Nemotron Kaggle adapter-only challenge.
Confirmed safe baseline V194: all720={V194_ALL720}, official360={V194_OFFICIAL360}.
Recent candidates A/C improved all720 but regressed official360. Regressions concentrate in cryptarithm_guess, cryptarithm_deduce, cipher.
Task: propose one surgical next experiment to beat V194 on both gates. Prefer solver-verified data and adapter triage. Reject broad micro-SFT.
Return JSON keys: verdict, next_experiment, required_data_filter, adapter_public_triage_policy, hard_rejects, promotion_gate.
'''.strip()

if RUN_OPENROUTER_PANEL:
    models = list(OPENROUTER_MODELS)
    if ALLOW_PAID_OPENROUTER:
        models += PAID_OPENROUTER_MODELS
    rows = []
    for model_id in models:
        started = time.time()
        try:
            data = openrouter_chat(model_id, audit_prompt)
            content = (data.get('choices') or [{}])[0].get('message', {}).get('content')
            rows.append({'model': model_id, 'ok': True, 'elapsed_sec': round(time.time() - started, 2), 'content': content, 'usage': data.get('usage')})
        except Exception as exc:
            rows.append({'model': model_id, 'ok': False, 'elapsed_sec': round(time.time() - started, 2), 'error': repr(exc)})
        print(rows[-1])
    out = OUT_DIR / 'V203_openrouter_panel.json'
    out.write_text(json.dumps(rows, indent=2, sort_keys=True), encoding='utf-8')
    print('Wrote:', out)
else:
    print('Skipped. Set RUN_OPENROUTER_PANEL=True after setting OPENROUTER_API_KEY.')


In [ ]:
roadmap = {
    'generated_at': utc_now(),
    'decision': 'KEEP_V194_AS_BASELINE',
    'submit_candidate': False,
    'baseline': {
        'label': 'V194',
        'adapter_dir': str(V194_ADAPTER),
        'adapter_model_sha256': V194_ADAPTER_SHA256,
        'all720': V194_ALL720,
        'official360': V194_OFFICIAL360,
    },
    'promotion_gate': [
        'candidate official360 loss must be strictly lower than V194 official360',
        'candidate all720 loss must be strictly lower than V194 all720',
        'preflight production_ready must be true',
        'no namespace/key-contract warning allowed',
        'no Kaggle submit from this notebook',
    ],
    'next_queue': [
        {
            'id': 'V203_public_inventory',
            'action': 'Use HF/Kaggle/OpenRouter inventory and save source metadata.',
            'status': 'prepared_by_this_notebook',
        },
        {
            'id': 'V204_public_adapter_triage',
            'action': 'Evaluate public HF adapters against official360 first, then both gates only if official360 beats V194.',
            'status': 'manual_flag_required',
        },
        {
            'id': 'V205_verified_trace_dataset',
            'action': 'Mine andy279 raw traces only when answer is solver-verified; focus cryptarithm_guess, cryptarithm_deduce, cipher.',
            'status': 'requires_HF_terms_token',
        },
        {
            'id': 'V206_micro_sft_from_verified_data',
            'action': 'Only after verified corpus exists; no broad Tong micro-SFT from V194.',
            'status': 'blocked_until_data_gate_passes',
        },
    ],
    'public_sources_to_track': inventory.get('known_public_refs', {}),
}

roadmap_path = OUT_DIR / 'V203_V204_PUBLIC_INTEL_ROADMAP.json'
roadmap_path.write_text(json.dumps(roadmap, indent=2, sort_keys=True), encoding='utf-8')
print('Wrote:', roadmap_path)
print(json.dumps(roadmap, indent=2))
